In [ ]:
#Shape and types


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

airports = pd.read_csv("../../data/raw/AA_AIRPORT_INFO.csv")
pax = pd.read_csv("../../data/raw/AA_PAX_FORECASTING.csv")

print("Airports:", airports.shape)
print("Pax:", pax.shape)

In [ ]:
for name, df in [("AA_AIRPORT_INFO", airports), ("AA_PAX_FORECASTING", pax)]:
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print(df.dtypes, "\n")

In [ ]:
display(airports.head())
display(pax.head())
display(pax.describe(include="all").T)

In [ ]:
#Column Name Meanings
#Airports:
#AIRPRT_CD: 3 LetterAirport code
#AIRPRT NM: Airport full name
#CITY_METRO_IATA_C: City metro IATA code, seperate from airport code
#STATE_PROVNC_CD: State or province code
#CNTRY_CD: Country code
#World Area Department of Transportation Code: 3 digit number that represents a geographic region
#LAT_DEGREE_QTY: Latitude of the airport in degrees
#LAT_MINUTE_QTY: Latitude of the airport in minutes 0-59
#LAT_SECOND_QTY: Latitude of the airport in seconds 0-59
#LAT_HEMSPHR_CD: Represents whether the latitude is in the Northern or Southern Hemisphere N or S
#LNGTD_DEGREE_QTY: Longitude of the airport in degrees
#LNGTD_MINUTE_QTY: Longitude of the airport in minutes 0-59
#LNGTD_SECOND_QTY: Longitude of the airport in seconds 0-59
#LNGTD_HEMSPHR_CD: Represents whether the longitude is in the Eastern or Western Hemisphere E or W
#LNGST_RUNWAY_FT_QTY: Length of the longest runway in feet
#ELEVATN_FT_QTY: Elevation of the airport in feet
#SF_LOAD_TMS: Snowflake Load Timestamp, represents the timestamp when the data was loaded into Snowflake

In [ ]:
#Column Name Meanings
#PAX
#DEP_MONTH: Month of departure
#DEP_YEAR: Year of departure
#ORG_AIRPRT: Origin airport code
#DEST_AIRPRT: Destination airport code
#OD_PAX: Total count of passangers from origin to destination including layovers and such
#SF_LOAD_TMS: Snowflake Load Timestamp, represents the timestamp when the data was loaded into Snowflake



In [ ]:
for name, df in [("airports", airports), ("pax", pax)]:
    miss = df.isna().mean().mul(100).round(2)
    print(f"{name} — % missing per column")
    print(miss[miss > 0].sort_values(ascending=False) if miss.any() else "No missing values", "\n")

In [ ]:
#Missingness
#The forecasting data set has missing values in 0 columns
#The airport data set has missing values in 2 columns, being the state-province-code and country code.
# Country Code missingness is minimal, but State/Province Code missingness is over 50%, so it should be dropped before modeling.

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

target = "OD_PAX"
data = pax[target].dropna()
cap = data.quantile(0.99)   # value below which 99% of route-months fall

sns.set_style("whitegrid")
plt.figure(figsize=(10, 5))
sns.histplot(data[data <= cap], bins=50, kde=True)
plt.title(f"Distribution of {target} (bottom 99%, tail above {cap:,.0f} not shown)")
plt.xlabel("Passengers")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

print(f"Median: {data.median():,.0f}  Mean: {data.mean():,.0f}  Max: {data.max():,.0f}  Skew: {data.skew():.2f}")

In [ ]:
#Distributions:
# OD_PAX is heavily right-skewed at 5.32.
# The median route-month has 503 passengers and the mean is 2765
# The max is 140194, so a small number of very busy routes pull the average way up



In [ ]:
print(pax.select_dtypes("number").corr()[target].sort_values(ascending=False).round(2))

In [ ]:
# Leakage
# No numeric column is highly correlated with the target (all below 0.95), so none appear to be derived from it.
# The airport table is static reference data, so joining it does not leak.

# Recommendation:
# Gradient-boosted trees XGBoost
# Reason: the target is heavily right-skewed skew 5.32 with a long tail of very busy routes, and tree splits are 
# not distorted by scale or outliers the way linear models are.